In [ ]:
#import
import os, sys
import json
from typing import Iterable, Tuple, List, Union
from pathlib import Path
import numpy as np
import pandas as pd
import subprocess
from scipy.ndimage import distance_transform_edt

sys.path.append('/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM')
from src.inference import (get_chains,
                            setup_folders,
                            strip_chain_data_from_pdb,
                            save_pdb,
                            save_grid_data,
                            F_exclude_near,
                            calc_voxel_distances,
                            write_bild_spheres,
                            sample_points,
                            clean_up)

In [ ]:
ROOT = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM")
BASE = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/negativeSamples/randomSampling")
PDB_ID = "3hfm"
RAW = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/data") / f"{PDB_ID}.pdb"
assert RAW.is_file()

params = {"VOXEL_SIZE" : 0.5,
          "PADDING" : 4}


#Get chains
ag_chains, ab_chains = get_chains(RAW)

#Setup AB and AG folder
output_dirs = setup_folders(base = BASE,
              pdb_id = PDB_ID,
              exist_ok= True,
              parents = True)


In [3]:
for output_dir, chains  in zip(output_dirs, [ag_chains, ab_chains]):
    
    data_stripped = strip_chain_data_from_pdb(pdb_path= RAW,
                                     target_chains=chains)


    
    save_pdb(data_stripped = data_stripped,
            output_path = output_dir / "strip_pdb.pdb")
    
    grid = save_grid_data(data =  data_stripped,
                   voxel_size = params["VOXEL_SIZE"],
                   padding =  params["PADDING"],
                   output_dir = output_dir)


    #Run open3d code
    subprocess.run([
        "conda", "run", "-n", "open3d_env",
        "python", f"{ROOT}/scripts/open3d_processing.py",
        output_dir
    ])

    surface_area = np.load(output_dir / "surface_area.npy")
    print(surface_area)
    distances = calc_voxel_distances(atom_cloud = data_stripped[["x", "y", "z"]],
                                    grid = grid,
                                    voxel_size = params["VOXEL_SIZE"])




    inside_mask = np.load(output_dir / "inside_mask.npy")
    inside_mask = inside_mask.reshape(distances.shape)
    distances[inside_mask] = 0 #exclude interior

    A_0 = 10000.
    n0 = 1000.
    b = 1.

    print(f"n_samples: {round(n0 * (surface_area / A_0) ** b)}")

    pts = sample_points(grid_box = grid,
                  grid_distances = distances,
                  n_samples = round(n0 * (surface_area / A_0) ** b),
                  mask = None,
                  weighing_func = F_exclude_near)

    write_bild_spheres(output_dir / "points.bild", points = pts, radius = 0.3)
    clean_up(dire= output_dir,
             files_to_delete= ["atom_cloud.npy", "voxel_centers.npy", "inside_mask.npy"])



unidentified atom type (hydrogen? belonging to protein) OXT


/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/scripts/open3d_processing.py:1: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  import open3d as o3d



5475.376957820271
n_samples: 548
unidentified atom type (hydrogen? belonging to protein) OXT
unidentified atom type (hydrogen? belonging to protein) OXT


/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/scripts/open3d_processing.py:1: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguious
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  import open3d as o3d



20753.35353394233
n_samples: 2075
